# OatEasy — Notebook 01: Synthetic Dataset Generation

Builds the three OatEasy datasets plus reference tables. **All data is synthetic** (simulated from explicit rules) because no real restaurant POS, inventory or waste logs are available. Models trained on it demonstrate the pipeline; they are not evidence of real-world accuracy.

| Dataset | Used by | File |
|---|---|---|
| 1 — Sales & demand | Model 1 (dish demand) | `data/dataset1_sales_demand.csv` |
| 2 — Inventory & waste | Model 2 (ingredient risk) | `data/dataset2_inventory_waste.csv` |
| 3 — Recipe & purchasing | Engine 3 (requirements/planning) | `data/dataset3_daily_requirements.csv`, `data/dataset3_purchase_orders.csv`, `data/dataset3_recipes.csv` |
| Reference | all | `data/dish_master.csv`, `data/ingredient_master.csv`, `data/restaurants.csv`, `data/calendar.csv` |

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
RNG = np.random.default_rng(SEED)
DATA = Path("../data")
DATA.mkdir(parents=True, exist_ok=True)

START, END = "2024-01-01", "2025-12-31"   # history window
FUTURE_END = "2026-01-31"                 # calendar extended so the next day can be forecast

## 1. Calendar, holidays and weather climatology

In [2]:
HOLIDAYS = {
    "2024-01-01": "New Year", "2024-01-26": "Republic Day", "2024-03-25": "Holi",
    "2024-08-15": "Independence Day", "2024-08-19": "Raksha Bandhan", "2024-10-12": "Dussehra",
    "2024-10-31": "Diwali", "2024-12-25": "Christmas",
    "2025-01-01": "New Year", "2025-01-26": "Republic Day", "2025-03-14": "Holi",
    "2025-08-09": "Raksha Bandhan", "2025-08-15": "Independence Day", "2025-10-02": "Dussehra",
    "2025-10-20": "Diwali", "2025-12-25": "Christmas",
    "2026-01-01": "New Year", "2026-01-26": "Republic Day",
}
MONTH_FACTOR = {1: 0.95, 2: 0.97, 3: 1.00, 4: 1.00, 5: 0.98, 6: 0.97,
                7: 0.95, 8: 0.96, 9: 0.98, 10: 1.08, 11: 1.10, 12: 1.05}
DOW_FACTOR = {"Monday": 0.90, "Tuesday": 0.92, "Wednesday": 0.95, "Thursday": 1.00,
              "Friday": 1.08, "Saturday": 1.00, "Sunday": 0.98}

def temp_climatology(doy):
    # simplified seasonal curve in deg C: ~14 in mid-January, ~36 in mid-July
    return 25 + 11 * np.sin(2 * np.pi * (np.asarray(doy) - 105) / 365)

cal = pd.DataFrame({"date": pd.date_range(START, FUTURE_END, freq="D")})
cal["day_of_week"] = cal["date"].dt.day_name()
cal["dow"] = cal["date"].dt.dayofweek
cal["is_weekend"] = (cal["dow"] >= 5).astype(int)
cal["month"] = cal["date"].dt.month
cal["holiday_name"] = cal["date"].dt.strftime("%Y-%m-%d").map(HOLIDAYS).fillna("")
cal["holiday_flag"] = (cal["holiday_name"] != "").astype(int)
cal["temp_climatology_c"] = temp_climatology(cal["date"].dt.dayofyear).round(1)
cal.to_csv(DATA / "calendar.csv", index=False)
cal.head()

,date,day_of_week,dow,is_weekend,month,holiday_name,holiday_flag,temp_climatology_c
0,2024-01-01,Monday,0,0,1,New Year,1,14.3
1,2024-01-02,Tuesday,1,0,1,,0,14.2
2,2024-01-03,Wednesday,2,0,1,,0,14.2
3,2024-01-04,Thursday,3,0,1,,0,14.2
4,2024-01-05,Friday,4,0,1,,0,14.1


## 2. Restaurants, dish menu, ingredients and recipes

In [3]:
restaurants = pd.DataFrame([
    {"restaurant_id": "R01", "restaurant_name": "Restaurant A", "demand_multiplier": 1.00},
    {"restaurant_id": "R02", "restaurant_name": "Restaurant B", "demand_multiplier": 0.80},
])

dish_cols = ["dish_id", "dish_name", "category", "list_price_inr",
             "base_daily_orders", "weekend_uplift", "temp_sensitivity"]
dishes = pd.DataFrame([
    ("D01", "Paneer Tikka",        "Starter",          320, 30, 0.25,  0.000),
    ("D02", "Dal Makhani",         "Main Course",      280, 26, 0.15, -0.010),
    ("D03", "Veg Biryani",         "Rice & Biryani",   340, 40, 0.35,  0.000),
    ("D04", "Butter Naan",         "Breads",            60, 55, 0.10,  0.000),
    ("D05", "Chole Bhature",       "Main Course",      220, 22, 0.30, -0.008),
    ("D06", "Palak Paneer",        "Main Course",      300, 20, 0.10, -0.012),
    ("D07", "Jeera Rice",          "Rice & Biryani",   150, 25, 0.10,  0.000),
    ("D08", "Masala Dosa",         "South Indian",     180, 24, 0.20,  0.006),
    ("D09", "Veg Thali",           "Combos",           420, 18, 0.30,  0.000),
    ("D10", "Samosa Chaat",        "Starter",          140, 16, 0.20,  0.010),
    ("D11", "Gobi Manchurian",     "Starter",          230, 14, 0.25,  0.000),
    ("D12", "Mushroom Pepper Fry", "Starter",          310,  9, 0.20,  0.000),
], columns=dish_cols)

# ingredient: id, name, category, unit, unit_cost, shelf_life_days, storage, perishable, lead_time_days, moq, target_cover_days, supplier
ing_cols = ["ingredient_id", "ingredient_name", "category", "unit", "unit_cost_inr", "shelf_life_days",
            "storage_type", "perishable_flag", "supplier_lead_time_days", "min_order_qty",
            "target_cover_days", "supplier_name"]
ingredients = pd.DataFrame([
    ("I01", "Paneer",                "Dairy",   "kg",     380,   7, "Chiller",   1, 1,  2, 3, "Dairy Co-op"),
    ("I02", "Tomato",                "Produce", "kg",      40,   5, "Cool room", 1, 1,  5, 3, "Local Mandi"),
    ("I03", "Onion",                 "Produce", "kg",      35,  21, "Dry store", 0, 1, 10, 7, "Local Mandi"),
    ("I04", "Fresh Cream",           "Dairy",   "kg",     420,   5, "Chiller",   1, 1,  2, 4, "Dairy Co-op"),
    ("I05", "Butter",                "Dairy",   "kg",     520,  60, "Chiller",   1, 2,  5, 14, "Dairy Co-op"),
    ("I06", "Wheat Flour (Atta)",    "Dry",     "kg",      48, 180, "Dry store", 0, 3, 25, 14, "Grain Traders"),
    ("I07", "Basmati Rice",          "Dry",     "kg",     110, 365, "Dry store", 0, 3, 25, 14, "Grain Traders"),
    ("I08", "Urad Dal",              "Dry",     "kg",     130, 180, "Dry store", 0, 3, 15, 14, "Grain Traders"),
    ("I09", "Chickpeas (Kabuli)",    "Dry",     "kg",     120, 180, "Dry store", 0, 3, 15, 14, "Grain Traders"),
    ("I10", "Spinach",               "Produce", "kg",      60,   3, "Cool room", 1, 1,  3, 2, "Local Mandi"),
    ("I11", "Cauliflower",           "Produce", "kg",      50,   6, "Cool room", 1, 1,  4, 3, "Local Mandi"),
    ("I12", "Potato",                "Produce", "kg",      30,  30, "Dry store", 0, 2, 10, 7, "Local Mandi"),
    ("I13", "Yogurt (Curd)",         "Dairy",   "kg",      90,   7, "Chiller",   1, 1,  3, 3, "Dairy Co-op"),
    ("I14", "Mushroom",              "Produce", "kg",     180,   4, "Cool room", 1, 2,  2, 2, "Fresh Farms"),
    ("I15", "Capsicum (Bell Pepper)","Produce", "kg",      90,   6, "Cool room", 1, 1,  3, 3, "Local Mandi"),
    ("I16", "Spice Mix (Garam Masala)","Dry",  "kg",     900, 365, "Dry store", 0, 5,  1, 30, "Spice Co."),
    ("I17", "Cooking Oil",           "Dry",     "litre", 160, 365, "Dry store", 0, 3, 15, 21, "Oil Distributor"),
], columns=ing_cols)

# recipes: dish_id -> {ingredient_id: (qty per portion, prep yield loss fraction)}
RECIPES = {
    "D01": {"I01": 0.10, "I13": 0.03, "I15": 0.02, "I03": 0.02, "I17": 0.01, "I16": 0.005},
    "D02": {"I08": 0.06, "I05": 0.01, "I04": 0.01, "I02": 0.03, "I03": 0.02, "I16": 0.004, "I17": 0.005},
    "D03": {"I07": 0.15, "I12": 0.03, "I03": 0.03, "I13": 0.03, "I17": 0.02, "I11": 0.04, "I16": 0.006},
    "D04": {"I06": 0.10, "I05": 0.02, "I13": 0.01, "I17": 0.005},
    "D05": {"I09": 0.10, "I03": 0.03, "I02": 0.03, "I06": 0.08, "I17": 0.03, "I16": 0.006},
    "D06": {"I10": 0.12, "I01": 0.08, "I04": 0.01, "I03": 0.02, "I02": 0.02, "I16": 0.004},
    "D07": {"I07": 0.12, "I05": 0.005, "I17": 0.01, "I16": 0.002},
    "D08": {"I07": 0.08, "I08": 0.03, "I12": 0.08, "I03": 0.02, "I17": 0.01},
    "D09": {"I01": 0.04, "I08": 0.03, "I07": 0.08, "I06": 0.05, "I10": 0.03, "I12": 0.04,
            "I13": 0.02, "I05": 0.005, "I17": 0.01},
    "D10": {"I12": 0.08, "I09": 0.03, "I13": 0.02, "I03": 0.02, "I06": 0.04, "I17": 0.03, "I16": 0.004},
    "D11": {"I11": 0.15, "I06": 0.02, "I15": 0.02, "I03": 0.02, "I17": 0.03, "I16": 0.005},
    "D12": {"I14": 0.12, "I15": 0.03, "I03": 0.03, "I17": 0.02, "I16": 0.005, "I05": 0.005},
}
RECIPE_LOSS = {  # prep yield loss per dish (fraction of raw qty that is not servable)
    "D01": 0.08, "D02": 0.05, "D03": 0.06, "D04": 0.03, "D05": 0.05, "D06": 0.09,
    "D07": 0.04, "D08": 0.05, "D09": 0.06, "D10": 0.07, "D11": 0.06, "D12": 0.05,
}
recipes = pd.DataFrame([
    {"dish_id": d, "ingredient_id": i,
     "qty_per_portion": q,
     "unit": ingredients.set_index("ingredient_id").loc[i, "unit"],
     "prep_yield_loss_pct": RECIPE_LOSS[d]}
    for d, items in RECIPES.items() for i, q in items.items()
])

restaurants.to_csv(DATA / "restaurants.csv", index=False)
dishes.to_csv(DATA / "dish_master.csv", index=False)
ingredients.to_csv(DATA / "ingredient_master.csv", index=False)
recipes.to_csv(DATA / "dataset3_recipes.csv", index=False)
print(len(dishes), "dishes,", len(ingredients), "ingredients,", len(recipes), "recipe lines")

12 dishes, 17 ingredients, 73 recipe lines


## 3. Dataset 1 — Sales & demand

Daily units sold per restaurant × dish. Demand is a Poisson draw around a rate driven by: base popularity, weekday/weekend, month (festive season), a slow growth trend, promotions (+20%, −10% price), public holidays (+25%), temperature sensitivity per dish, and rain (−6%).

In [4]:
hist_cal = cal[cal["date"] <= END].reset_index(drop=True)
n_days = len(hist_cal)

temp = temp_climatology(hist_cal["date"].dt.dayofyear) + RNG.normal(0, 2.0, n_days)
rain_p = np.where(hist_cal["month"].isin([7, 8, 9]), 0.35, 0.05)
rain = np.where(RNG.random(n_days) < rain_p, RNG.exponential(12, n_days), 0.0).round(1)

trend = 1 + 0.08 * ((hist_cal["date"] - pd.Timestamp(START)).dt.days.values / 365)
dow_f = hist_cal["day_of_week"].map(DOW_FACTOR).values
month_f = hist_cal["month"].map(MONTH_FACTOR).values
holiday_f = np.where(hist_cal["holiday_flag"] == 1, 1.25, 1.0)
rain_f = np.where(rain > 0, 0.94, 1.0)

# restaurant-wide promotion blocks of 3-5 days
promo = {}
for rid in restaurants["restaurant_id"]:
    flags = np.zeros(n_days, dtype=int)
    i = 0
    while i < n_days:
        if RNG.random() < 0.035:
            L = int(RNG.integers(3, 6))
            flags[i:i + L] = 1
            i += L
        else:
            i += 1
    promo[rid] = flags

frames = []
for _, r in restaurants.iterrows():
    pf = promo[r["restaurant_id"]]
    for _, d in dishes.iterrows():
        weekend_f = np.where(hist_cal["is_weekend"] == 1, 1 + d["weekend_uplift"], 1.0)
        promo_f = np.where(pf == 1, 1.20, 1.0)
        temp_f = np.exp(d["temp_sensitivity"] * (temp - 25))
        lam = (d["base_daily_orders"] * r["demand_multiplier"] * dow_f * weekend_f * month_f
               * trend * promo_f * holiday_f * temp_f * rain_f
               * np.exp(RNG.normal(0, 0.10, n_days)))
        units = RNG.poisson(lam)
        price = np.where(pf == 1, d["list_price_inr"] * 0.9, d["list_price_inr"])
        frames.append(pd.DataFrame({
            "date": hist_cal["date"].dt.strftime("%Y-%m-%d").values,
            "restaurant_id": r["restaurant_id"],
            "dish_id": d["dish_id"],
            "dish_name": d["dish_name"],
            "category": d["category"],
            "day_of_week": hist_cal["day_of_week"].values,
            "is_weekend": hist_cal["is_weekend"].values,
            "month": hist_cal["month"].values,
            "list_price_inr": d["list_price_inr"],
            "selling_price_inr": np.round(price, 2),
            "promo_flag": pf,
            "holiday_flag": hist_cal["holiday_flag"].values,
            "avg_temp_c": np.round(temp, 1),
            "rainfall_mm": rain,
            "units_sold": units,
        }))

sales = (pd.concat(frames, ignore_index=True)
           .sort_values(["date", "restaurant_id", "dish_id"])
           .reset_index(drop=True))
sales.to_csv(DATA / "dataset1_sales_demand.csv", index=False)
print(sales.shape)
sales.head()

(17544, 15)


,date,restaurant_id,dish_id,dish_name,category,day_of_week,is_weekend,month,list_price_inr,selling_price_inr,promo_flag,holiday_flag,avg_temp_c,rainfall_mm,units_sold
0,2024-01-01,R01,D01,Paneer Tikka,Starter,Monday,0,1,320,320.0,0,1,14.9,0.0,26
1,2024-01-01,R01,D02,Dal Makhani,Main Course,Monday,0,1,280,280.0,0,1,14.9,0.0,37
2,2024-01-01,R01,D03,Veg Biryani,Rice & Biryani,Monday,0,1,340,340.0,0,1,14.9,0.0,49
3,2024-01-01,R01,D04,Butter Naan,Breads,Monday,0,1,60,60.0,0,1,14.9,0.0,55
4,2024-01-01,R01,D05,Chole Bhature,Main Course,Monday,0,1,220,220.0,0,1,14.9,0.0,30


## 4. Dataset 2 — Inventory & waste (batch-level simulation)

Each restaurant runs a simple ordering policy per ingredient: when projected stock falls below a reorder point, it orders up to `lead time + target cover` days of average usage, rounded to the supplier's minimum order quantity, with occasional one-day delivery delays. Stock is held in batches with a remaining shelf life; usage is taken first-expiring-first-out. Discards come from three sources: **expired** batches, **over-preparation** (perishables prepared beyond demand), and **spoilage**. Usage is derived from the sales in Dataset 1 and the recipes, so the datasets reconcile.

The stock balance is checked explicitly at the end: `opening + received − expired − spoilage − usage − over-prep = closing`.

In [5]:
dish_ids = dishes["dish_id"].tolist()
ing_ids = ingredients["ingredient_id"].tolist()
R = np.zeros((len(dish_ids), len(ing_ids)))       # raw qty per portion
R_eff = np.zeros_like(R)                           # qty incl. prep yield loss
for _, row in recipes.iterrows():
    i, j = dish_ids.index(row.dish_id), ing_ids.index(row.ingredient_id)
    R[i, j] = row.qty_per_portion
    R_eff[i, j] = row.qty_per_portion / (1 - row.prep_yield_loss_pct)

date_strs = hist_cal["date"].dt.strftime("%Y-%m-%d").values
START_TS = pd.Timestamp(START)

def date_of(idx):
    return START_TS + pd.to_timedelta(idx, unit="D")

def withdraw(batches, qty):
    """Take qty from batches, earliest expiry first. Returns (taken, shortfall)."""
    taken = 0.0
    for b in sorted(batches, key=lambda b: b["expiry"]):
        if qty <= 1e-9:
            break
        take = min(b["qty"], qty)
        b["qty"] -= take
        qty -= take
        taken += take
    batches[:] = [b for b in batches if b["qty"] > 1e-9]
    return taken, max(qty, 0.0)

def simulate_restaurant(rid, units_matrix, rng):
    need_all = (units_matrix @ R_eff) * rng.normal(1.0, 0.03, size=(n_days, len(ing_ids)))
    expected_eff = units_matrix @ R_eff
    theo = units_matrix @ R
    rec_rows, req_rows, order_rows = [], [], []
    for j, ing in ingredients.iterrows():
        batches, pending, usage_hist = [], [], []
        last_delivery = None
        lead, moq = int(ing["supplier_lead_time_days"]), float(ing["min_order_qty"])
        cover, shelf = int(ing["target_cover_days"]), int(ing["shelf_life_days"])
        perish = int(ing["perishable_flag"])
        for t in range(n_days):
            opening = sum(b["qty"] for b in batches)
            arrivals = [o for o in pending if o["arrive"] == t]
            pending = [o for o in pending if o["arrive"] != t]
            received = 0.0
            for o in arrivals:
                life = max(1, int(round(shelf * rng.uniform(0.6, 1.0))))
                batches.append({"qty": o["qty"], "expiry": t + life})
                received += o["qty"]
                o["actual_delivery_idx"] = t
                last_delivery = t
            expired = sum(b["qty"] for b in batches if b["expiry"] < t)
            batches[:] = [b for b in batches if b["expiry"] >= t]
            spoil = 0.0
            if perish and rng.random() < 0.02:
                spoil, _ = withdraw(batches, 0.05 * sum(b["qty"] for b in batches))
            need = need_all[t, j]
            taken, short = withdraw(batches, need)
            # over-preparation is an occasional event (~30% of days), not a daily certainty
            overprep = (need * max(0.0, rng.normal(0.05, 0.03))
                        if (perish and rng.random() < 0.30) else 0.0)
            overprep_taken, _ = withdraw(batches, overprep)
            closing = sum(b["qty"] for b in batches)

            usage_hist.append(need)   # ordering reacts to true demand, not stock-limited usage
            avg7 = float(np.mean(usage_hist[-7:]))
            pending_qty = sum(o["qty"] for o in pending)
            ordered = 0.0
            if avg7 > 0 and (closing + pending_qty) < avg7 * (lead + 1):
                target = avg7 * (lead + cover)
                q = max(target - (closing + pending_qty), moq)
                q = float(np.ceil(q / moq) * moq)
                delay = 1 if rng.random() < 0.10 else 0
                o = {"order_id": f"PO-{rid}-{ing['ingredient_id']}-{t:04d}",
                     "restaurant_id": rid, "ingredient_id": ing["ingredient_id"],
                     "ingredient_name": ing["ingredient_name"], "order_idx": t,
                     "expected_idx": t + lead, "arrive": t + lead + delay,
                     "qty": q, "unit_cost_inr": ing["unit_cost_inr"],
                     "planned_lead_time_days": lead, "actual_delivery_idx": None}
                pending.append(o)
                order_rows.append(o)
                ordered = q
                pending_qty += q

            mins = [b["expiry"] - t for b in batches]
            min_days = min(mins) if mins else np.nan
            wavg = (sum(b["qty"] * (b["expiry"] - t) for b in batches) / closing) if closing > 1e-9 else np.nan
            parts = {"expired": expired, "overprep": overprep_taken, "spoilage": spoil}
            total = sum(parts.values())
            reason = max(parts, key=parts.get) if total > 1e-9 else "none"
            rec_rows.append({
                "date": date_strs[t], "restaurant_id": rid, "ingredient_id": ing["ingredient_id"],
                "ingredient_name": ing["ingredient_name"], "category": ing["category"],
                "perishable_flag": perish, "storage_type": ing["storage_type"], "unit": ing["unit"],
                "unit_cost_inr": ing["unit_cost_inr"],
                "opening_stock_qty": round(opening, 3), "received_qty": round(received, 3),
                "actual_usage_qty": round(taken, 3), "shortage_qty": round(short, 3),
                "expired_discard_qty": round(expired, 3), "overprep_discard_qty": round(overprep_taken, 3),
                "spoilage_discard_qty": round(spoil, 3), "total_discard_qty": round(total, 3),
                "discard_reason": reason, "discard_cost_inr": round(total * ing["unit_cost_inr"], 2),
                "closing_stock_qty": round(closing, 3), "batches_on_hand": len(batches),
                "min_days_to_expiry": min_days, "weighted_remaining_shelf_life_days": wavg,
                "days_since_last_delivery": min(t - last_delivery, 365) if last_delivery is not None else 365,
                "ordered_qty_today": round(ordered, 3), "pending_order_qty": round(pending_qty, 3),
            })
            req_rows.append({
                "date": date_strs[t], "restaurant_id": rid, "ingredient_id": ing["ingredient_id"],
                "ingredient_name": ing["ingredient_name"], "unit": ing["unit"],
                "theoretical_requirement_qty": round(theo[t, j], 3),
                "planned_requirement_with_yield_loss_qty": round(expected_eff[t, j], 3),
                "actual_required_qty": round(need, 3), "actual_consumed_qty": round(taken, 3),
            })
    # finalise orders: arrival beyond the history window is still in transit
    for o in order_rows:
        delivered = o["actual_delivery_idx"] is not None and o["actual_delivery_idx"] < n_days
        order_rows_fmt = None
    orders_df = pd.DataFrame([{
        "order_id": o["order_id"], "restaurant_id": o["restaurant_id"],
        "ingredient_id": o["ingredient_id"], "ingredient_name": o["ingredient_name"],
        "order_date": date_of(o["order_idx"]).strftime("%Y-%m-%d"),
        "expected_delivery_date": date_of(o["expected_idx"]).strftime("%Y-%m-%d"),
        "actual_delivery_date": (date_of(o["actual_delivery_idx"]).strftime("%Y-%m-%d")
                                 if o["actual_delivery_idx"] is not None and o["actual_delivery_idx"] < n_days else None),
        "planned_lead_time_days": o["planned_lead_time_days"],
        "actual_lead_time_days": (o["actual_delivery_idx"] - o["order_idx"]
                                  if o["actual_delivery_idx"] is not None and o["actual_delivery_idx"] < n_days else None),
        "qty_ordered": o["qty"],
        "qty_received": o["qty"] if (o["actual_delivery_idx"] is not None and o["actual_delivery_idx"] < n_days) else 0.0,
        "unit_cost_inr": o["unit_cost_inr"],
        "purchase_value_inr": round(o["qty"] * o["unit_cost_inr"], 2),
        "status": "received" if (o["actual_delivery_idx"] is not None and o["actual_delivery_idx"] < n_days) else "in_transit",
    } for o in order_rows])
    return pd.DataFrame(rec_rows), pd.DataFrame(req_rows), orders_df

inv_parts, req_parts, order_parts = [], [], []
for rid in restaurants["restaurant_id"]:
    units_matrix = (sales[sales["restaurant_id"] == rid]
                    .pivot(index="date", columns="dish_id", values="units_sold")
                    .reindex(index=date_strs, columns=dish_ids).values.astype(float))
    inv_r, req_r, ord_r = simulate_restaurant(rid, units_matrix, RNG)
    inv_parts.append(inv_r); req_parts.append(req_r); order_parts.append(ord_r)

inventory = pd.concat(inv_parts, ignore_index=True).sort_values(["date", "restaurant_id", "ingredient_id"]).reset_index(drop=True)
requirements = pd.concat(req_parts, ignore_index=True).sort_values(["date", "restaurant_id", "ingredient_id"]).reset_index(drop=True)
orders = pd.concat(order_parts, ignore_index=True).sort_values(["order_date", "restaurant_id", "ingredient_id"]).reset_index(drop=True)

# stock balance identity check
resid = (inventory["opening_stock_qty"] + inventory["received_qty"]
         - inventory["expired_discard_qty"] - inventory["spoilage_discard_qty"]
         - inventory["actual_usage_qty"] - inventory["overprep_discard_qty"]
         - inventory["closing_stock_qty"])
print("max stock balance residual:", round(resid.abs().max(), 4))
assert resid.abs().max() < 0.01, "stock balance broken"

inventory.to_csv(DATA / "dataset2_inventory_waste.csv", index=False)
requirements.to_csv(DATA / "dataset3_daily_requirements.csv", index=False)
orders.to_csv(DATA / "dataset3_purchase_orders.csv", index=False)
print("dataset2 rows:", inventory.shape, "| dataset3 requirements:", requirements.shape, "| purchase orders:", orders.shape)
print("discard share of total stock-in:",
      round(inventory["total_discard_qty"].sum() / (inventory["received_qty"].sum() + 1e-9), 4))
inventory.head()

max stock balance residual: 0.001


dataset2 rows: (24854, 26) | dataset3 requirements: (24854, 9) | purchase orders: (5100, 14)
discard share of total stock-in: 0.0201


,date,restaurant_id,ingredient_id,ingredient_name,category,perishable_flag,storage_type,unit,unit_cost_inr,opening_stock_qty,...,total_discard_qty,discard_reason,discard_cost_inr,closing_stock_qty,batches_on_hand,min_days_to_expiry,weighted_remaining_shelf_life_days,days_since_last_delivery,ordered_qty_today,pending_order_qty
0,2024-01-01,R01,I01,Paneer,Dairy,1,Chiller,kg,380,0.0,...,0.0,none,0.0,0.0,0,NaN,NaN,365,24.0,24.0
1,2024-01-01,R01,I02,Tomato,Produce,1,Cool room,kg,40,0.0,...,0.0,none,0.0,0.0,0,NaN,NaN,365,15.0,15.0
2,2024-01-01,R01,I03,Onion,Produce,0,Dry store,kg,35,0.0,...,0.0,none,0.0,0.0,0,NaN,NaN,365,50.0,50.0
3,2024-01-01,R01,I04,Fresh Cream,Dairy,1,Chiller,kg,420,0.0,...,0.0,none,0.0,0.0,0,NaN,NaN,365,4.0,4.0
4,2024-01-01,R01,I05,Butter,Dairy,1,Chiller,kg,520,0.0,...,0.0,none,0.0,0.0,0,NaN,NaN,365,35.0,35.0


## 5. Summary

In [6]:
print("Files written to", DATA.resolve())
for f in sorted(DATA.glob("*.csv")):
    print(f"{f.name:40s} {sum(1 for _ in open(f)) - 1:>8,d} rows")

Files written to /home/claude/OatEasy_ML/data
calendar.csv                                  762 rows
dataset1_sales_demand.csv                  17,544 rows
dataset2_inventory_waste.csv               24,854 rows
dataset3_daily_requirements.csv            24,854 rows
dataset3_purchase_orders.csv                5,100 rows
dataset3_recipes.csv                           73 rows
dish_master.csv                                12 rows
ingredient_master.csv                          17 rows
restaurants.csv                                 2 rows
